# Quantis — Data Pipeline (Colab)

Run this notebook to refresh the data on your live Quantis dashboard.

**What it does, in order:**
1. Installs dependencies
2. Defines the 100-ticker universe (edit this cell to add/remove tickers)
3. Authenticates to GitHub using a token stored in Colab Secrets
4. **Bars pipeline** — pulls 5y of OHLCV from Yahoo, writes `data/bars/<TICKER>.json` (~2 min for 100 tickers)
5. **Forecasts pipeline** — trains LSTM + ARIMA + RF for each ticker, writes `data/forecasts/<TICKER>.json` (~30–90 min depending on whether you're on a GPU runtime)
6. Pushes everything to your GitHub repo in one commit, GitHub Pages redeploys in ~30 seconds

**One-time setup:** see the next cell. After that you can re-run the whole notebook (Runtime → Run all) anytime to refresh.

## One-time setup

**1. GitHub Personal Access Token (PAT)**  
Go to https://github.com/settings/tokens → "Fine-grained tokens" → "Generate new token". Grant it access to **only your `quantis` repo**, with **Contents: Read and write** permission. Copy the token.

**2. Store the token in Colab Secrets**  
In Colab, click the 🔑 key icon in the left sidebar → "Add new secret" → Name: `GITHUB_TOKEN`, Value: paste your token → toggle "Notebook access" on.

**3. Set your repo coordinates in the next cell** (`GH_USER`, `GH_REPO`, `GH_BRANCH`).

**4. Tip:** for the forecasts pipeline, switch to a GPU runtime (Runtime → Change runtime type → T4 GPU). Free GPUs make LSTM training ~10× faster.

In [ ]:
# ============================================================
# CONFIG — edit these for your repo
# ============================================================

GH_USER   = 'amma-ctrl'       # your GitHub username
GH_REPO   = 'quantis'         # your repo name (case-sensitive)
GH_BRANCH = 'main'            # branch to push to (usually 'main')

# What to run (toggle these to skip sections)
RUN_BARS      = True          # fast (~2 min)
RUN_FORECASTS = True          # slow (~30-90 min)
PUSH_TO_GITHUB = True         # set False to inspect locally first

# Default: refresh every ticker in the 100-name universe.
# To run on a subset, set this to e.g. ['AAPL', 'MSFT', 'NVDA']
TICKERS_TO_RUN = None         # None = all 100

## 1. Install dependencies

In [ ]:
!pip install -q yfinance ta statsmodels PyGithub
# tensorflow + scikit-learn + pandas + numpy are preinstalled on Colab

## 2. Ticker universe (100 names)

If you want to add or remove tickers, edit the list below and re-run from here on. The same list is used for both the bars and forecasts pipelines.

In [ ]:
# (symbol, name, sector) — 100 US stocks & ETFs
TICKER_UNIVERSE = [
    # Mega-cap tech
    ('AAPL',  'Apple Inc.',                    'Tech'),
    ('MSFT',  'Microsoft Corporation',         'Tech'),
    ('GOOGL', 'Alphabet Inc. Class A',         'Tech'),
    ('GOOG',  'Alphabet Inc. Class C',         'Tech'),
    ('AMZN',  'Amazon.com Inc.',               'Tech'),
    ('NVDA',  'NVIDIA Corporation',            'Tech'),
    ('META',  'Meta Platforms Inc.',           'Tech'),
    ('TSLA',  'Tesla Inc.',                    'Auto'),
    # Semiconductors
    ('AVGO',  'Broadcom Inc.',                 'Tech'),
    ('AMD',   'Advanced Micro Devices',        'Tech'),
    ('QCOM',  'Qualcomm Inc.',                 'Tech'),
    ('TXN',   'Texas Instruments Inc.',        'Tech'),
    ('MU',    'Micron Technology Inc.',        'Tech'),
    ('TSM',   'Taiwan Semiconductor',          'Tech'),
    ('INTC',  'Intel Corp.',                   'Tech'),
    ('ARM',   'Arm Holdings plc',              'Tech'),
    # Software / SaaS
    ('CRM',   'Salesforce Inc.',               'Tech'),
    ('ORCL',  'Oracle Corp.',                  'Tech'),
    ('ADBE',  'Adobe Inc.',                    'Tech'),
    ('NOW',   'ServiceNow Inc.',               'Tech'),
    ('PLTR',  'Palantir Technologies',         'Tech'),
    ('SNOW',  'Snowflake Inc.',                'Tech'),
    ('SHOP',  'Shopify Inc.',                  'Tech'),
    ('UBER',  'Uber Technologies Inc.',        'Tech'),
    # Financials
    ('BRK-B', 'Berkshire Hathaway Inc.',       'Finance'),
    ('JPM',   'JPMorgan Chase & Co.',          'Finance'),
    ('BAC',   'Bank of America Corp.',         'Finance'),
    ('WFC',   'Wells Fargo & Co.',             'Finance'),
    ('C',     'Citigroup Inc.',                'Finance'),
    ('GS',    'Goldman Sachs Group Inc.',      'Finance'),
    ('MS',    'Morgan Stanley',                'Finance'),
    ('V',     'Visa Inc.',                     'Finance'),
    ('MA',    'Mastercard Inc.',               'Finance'),
    ('AXP',   'American Express Co.',          'Finance'),
    ('BLK',   'BlackRock Inc.',                'Finance'),
    ('SCHW',  'Charles Schwab Corp.',          'Finance'),
    # Healthcare
    ('UNH',   'UnitedHealth Group Inc.',       'Health'),
    ('JNJ',   'Johnson & Johnson',             'Health'),
    ('LLY',   'Eli Lilly and Co.',             'Health'),
    ('PFE',   'Pfizer Inc.',                   'Health'),
    ('MRK',   'Merck & Co. Inc.',              'Health'),
    ('ABBV',  'AbbVie Inc.',                   'Health'),
    ('TMO',   'Thermo Fisher Scientific',      'Health'),
    ('ABT',   'Abbott Laboratories',           'Health'),
    ('DHR',   'Danaher Corp.',                 'Health'),
    # Consumer
    ('WMT',   'Walmart Inc.',                  'Consumer'),
    ('COST',  'Costco Wholesale Corp.',        'Consumer'),
    ('HD',    'Home Depot Inc.',               'Consumer'),
    ('LOW',   "Lowe's Companies Inc.",         'Consumer'),
    ('TGT',   'Target Corp.',                  'Consumer'),
    ('NKE',   'Nike Inc.',                     'Consumer'),
    ('SBUX',  'Starbucks Corp.',               'Consumer'),
    ('MCD',   "McDonald's Corp.",              'Consumer'),
    ('PG',    'Procter & Gamble Co.',          'Consumer'),
    ('KO',    'Coca-Cola Co.',                 'Consumer'),
    ('PEP',   'PepsiCo Inc.',                  'Consumer'),
    ('ABNB',  'Airbnb Inc.',                   'Consumer'),
    ('BKNG',  'Booking Holdings Inc.',         'Consumer'),
    ('MAR',   'Marriott International',        'Consumer'),
    # Media / Telecom
    ('DIS',   'Walt Disney Co.',               'Media'),
    ('NFLX',  'Netflix Inc.',                  'Media'),
    ('CMCSA', 'Comcast Corp.',                 'Media'),
    ('T',     'AT&T Inc.',                     'Media'),
    ('VZ',    'Verizon Communications',        'Media'),
    ('SPOT',  'Spotify Technology',            'Media'),
    ('ROKU',  'Roku Inc.',                     'Media'),
    # Energy
    ('XOM',   'Exxon Mobil Corp.',             'Energy'),
    ('CVX',   'Chevron Corp.',                 'Energy'),
    ('COP',   'ConocoPhillips',                'Energy'),
    # Industrials
    ('BA',    'Boeing Co.',                    'Industry'),
    ('CAT',   'Caterpillar Inc.',              'Industry'),
    ('HON',   'Honeywell International',       'Industry'),
    ('UPS',   'United Parcel Service',         'Industry'),
    ('LMT',   'Lockheed Martin Corp.',         'Industry'),
    ('RTX',   'RTX Corp.',                     'Industry'),
    ('GE',    'GE Aerospace',                  'Industry'),
    ('DE',    'Deere & Co.',                   'Industry'),
    # Auto / EV
    ('F',     'Ford Motor Co.',                'Auto'),
    ('GM',    'General Motors Co.',            'Auto'),
    ('RIVN',  'Rivian Automotive Inc.',        'Auto'),
    ('LCID',  'Lucid Group Inc.',              'Auto'),
    ('NIO',   'NIO Inc.',                      'Auto'),
    # Fintech / Payments
    ('PYPL',  'PayPal Holdings Inc.',          'Finance'),
    ('COIN',  'Coinbase Global Inc.',          'Finance'),
    ('SQ',    'Block Inc.',                    'Finance'),
    # Misc tech
    ('CSCO',  'Cisco Systems Inc.',            'Tech'),
    ('IBM',   'IBM Corp.',                     'Tech'),
    ('DELL',  'Dell Technologies Inc.',        'Tech'),
    ('SNAP',  'Snap Inc.',                     'Tech'),
    ('RBLX',  'Roblox Corp.',                  'Tech'),
    ('BABA',  'Alibaba Group Holding',         'Tech'),
    ('JD',    'JD.com Inc.',                   'Tech'),
    # ETFs
    ('SPY',   'SPDR S&P 500 ETF',              'ETF'),
    ('QQQ',   'Invesco QQQ Trust',             'ETF'),
    ('VOO',   'Vanguard S&P 500 ETF',          'ETF'),
    ('VTI',   'Vanguard Total Stock Market',   'ETF'),
    ('IWM',   'iShares Russell 2000 ETF',      'ETF'),
    ('DIA',   'SPDR Dow Jones ETF',            'ETF'),
    ('GLD',   'SPDR Gold Shares',              'ETF'),
    ('SLV',   'iShares Silver Trust',          'ETF'),
]

NAMES = {sym: name for sym, name, _ in TICKER_UNIVERSE}
ALL_SYMBOLS = [t[0] for t in TICKER_UNIVERSE]
SYMBOLS = TICKERS_TO_RUN or ALL_SYMBOLS

print(f'{len(TICKER_UNIVERSE)} tickers in universe; running {len(SYMBOLS)} this session.')

## 3. Authenticate to GitHub

In [ ]:
from google.colab import userdata
from github import Github, Auth, InputGitTreeElement
import base64

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
assert GITHUB_TOKEN, 'Missing GITHUB_TOKEN secret. See setup instructions at the top.'

gh = Github(auth=Auth.Token(GITHUB_TOKEN))
repo = gh.get_repo(f'{GH_USER}/{GH_REPO}')
print(f'Authenticated as {gh.get_user().login}. Repo: {repo.full_name} ({repo.default_branch})')

## 4. Bars pipeline (~2 min for 100 tickers)

Pulls 5 years of daily OHLCV from Yahoo Finance. Writes one JSON per ticker to a local `out/data/bars/` folder. Nothing pushed yet — that happens in step 6.

In [ ]:
import json, time
from datetime import datetime, timezone
from pathlib import Path
import yfinance as yf

BARS_DIR = Path('out/data/bars')
BARS_DIR.mkdir(parents=True, exist_ok=True)

def fetch_bars(ticker, period='5y'):
    df = yf.Ticker(ticker).history(period=period, auto_adjust=False)
    if df.empty:
        raise ValueError(f'no data for {ticker}')
    bars = []
    for ts, row in df.iterrows():
        v = int(row['Volume']) if row['Volume'] == row['Volume'] else 0
        bars.append({
            'date':   ts.strftime('%Y-%m-%d'),
            'open':   round(float(row['Open']),  4),
            'high':   round(float(row['High']),  4),
            'low':    round(float(row['Low']),   4),
            'close':  round(float(row['Close']), 4),
            'volume': v,
        })
    return bars

bars_files = []  # list of (relative_path, content_str) to push later
bars_manifest_tickers = []

if RUN_BARS:
    print(f'Fetching bars for {len(SYMBOLS)} tickers...')
    t0 = time.time()
    for i, sym in enumerate(SYMBOLS, 1):
        try:
            bars = fetch_bars(sym)
            payload = {
                'ticker': sym,
                'name': NAMES.get(sym, sym),
                'generated_at': datetime.now(timezone.utc).isoformat(),
                'bars': bars,
            }
            content = json.dumps(payload, separators=(',', ':'))
            (BARS_DIR / f'{sym}.json').write_text(content)
            bars_files.append((f'data/bars/{sym}.json', content))
            bars_manifest_tickers.append(sym)
            print(f'  [{i:3}/{len(SYMBOLS)}] {sym:6} {len(bars)} bars')
        except Exception as e:
            print(f'  [{i:3}/{len(SYMBOLS)}] {sym:6} FAILED: {e}')

    manifest = {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'tickers': bars_manifest_tickers,
    }
    manifest_content = json.dumps(manifest, indent=2)
    (BARS_DIR / 'manifest.json').write_text(manifest_content)
    bars_files.append(('data/bars/manifest.json', manifest_content))

    print(f'\nBars: wrote {len(bars_manifest_tickers)}/{len(SYMBOLS)} tickers in {time.time()-t0:.0f}s')
else:
    print('Skipping bars (RUN_BARS=False).')

## 5. Forecasts pipeline (~30–90 min depending on runtime)

Trains the five models per ticker:
- Logistic Regression, Decision Tree, Random Forest (next-day direction)
- LSTM (3 stacked layers, 60-day lookback, 30-day forecast)
- ARIMA(5,1,0) (walk-forward + 30-day forecast)
- Multi-horizon RF (1, 3, 5, 10, 20 days)

On a free Colab GPU (T4) this runs in roughly 30–45 minutes for 100 tickers. On CPU expect closer to 90.

**To skip this section** and just refresh prices, set `RUN_FORECASTS = False` in the config cell at the top.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import ta
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from statsmodels.tsa.arima.model import ARIMA
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential

FEATURE_COLS = [
    'rsi','macd','macd_signal','macd_diff','stoch_k','stoch_d',
    'bb_width','bb_position','atr_percent','volume_ratio',
    'returns_1d','returns_5d','returns_10d','returns_20d',
    'volatility_20d','price_to_sma20','price_to_sma50',
    'daily_range','trend_strength',
]
LSTM_LOOKBACK = 60
LSTM_FORECAST_DAYS = 30
ARIMA_ORDER = (5, 1, 0)
ARIMA_FORECAST_DAYS = 30

FORECASTS_DIR = Path('out/data/forecasts')
FORECASTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def fetch_stock_data(ticker, period='5y'):
    df = yf.Ticker(ticker).history(period=period, auto_adjust=False)
    if df.empty:
        raise ValueError(f'no data for {ticker}')
    df.columns = [c.lower() for c in df.columns]
    df = df.reset_index()
    df['date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    return df[['date','open','high','low','close','volume']]

def engineer_features(df):
    df = df.copy()
    df['sma_20'] = ta.trend.sma_indicator(df['close'], window=20)
    df['sma_50'] = ta.trend.sma_indicator(df['close'], window=50)
    df['ema_12'] = ta.trend.ema_indicator(df['close'], window=12)
    df['ema_26'] = ta.trend.ema_indicator(df['close'], window=26)
    df['rsi'] = ta.momentum.rsi(df['close'], window=14)
    macd = ta.trend.MACD(df['close'])
    df['macd'] = macd.macd(); df['macd_signal'] = macd.macd_signal(); df['macd_diff'] = macd.macd_diff()
    stoch = ta.momentum.StochasticOscillator(df['high'], df['low'], df['close'])
    df['stoch_k'] = stoch.stoch(); df['stoch_d'] = stoch.stoch_signal()
    bb = ta.volatility.BollingerBands(df['close'])
    df['bb_high'] = bb.bollinger_hband(); df['bb_low'] = bb.bollinger_lband()
    df['bb_width'] = (df['bb_high'] - df['bb_low']) / bb.bollinger_mavg()
    df['bb_position'] = (df['close'] - df['bb_low']) / (df['bb_high'] - df['bb_low'])
    df['atr'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'])
    df['atr_percent'] = df['atr'] / df['close'] * 100
    df['volume_sma'] = df['volume'].rolling(20).mean()
    df['volume_ratio'] = df['volume'] / df['volume_sma']
    for n in (1, 5, 10, 20):
        df[f'returns_{n}d'] = df['close'].pct_change(n)
    df['volatility_20d'] = df['returns_1d'].rolling(20).std() * np.sqrt(252)
    df['price_to_sma20'] = df['close'] / df['sma_20']
    df['price_to_sma50'] = df['close'] / df['sma_50']
    df['daily_range'] = (df['high'] - df['low']) / df['close']
    df['trend_strength'] = abs(df['sma_20'] - df['sma_50']) / df['close']
    return df

def train_classifiers(df, prediction_days=1):
    df = df.copy()
    df['target'] = (df['close'].shift(-prediction_days) > df['close']).astype(int)
    df_clean = df.dropna(subset=FEATURE_COLS + ['target'])
    split = int(len(df_clean) * 0.8)
    Xtr, Xte = df_clean[FEATURE_COLS].iloc[:split].values, df_clean[FEATURE_COLS].iloc[split:].values
    ytr, yte = df_clean['target'].iloc[:split].values, df_clean['target'].iloc[split:].values
    sc = StandardScaler(); Xtr_s = sc.fit_transform(Xtr); Xte_s = sc.transform(Xte)
    results = {}
    lr = LogisticRegression(random_state=42, max_iter=1000).fit(Xtr_s, ytr)
    results['logistic_regression'] = {'accuracy': float(accuracy_score(yte, lr.predict(Xte_s)))}
    dt = DecisionTreeClassifier(random_state=42, max_depth=10).fit(Xtr_s, ytr)
    results['decision_tree'] = {'accuracy': float(accuracy_score(yte, dt.predict(Xte_s)))}
    rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10).fit(Xtr_s, ytr)
    rf_pred = rf.predict(Xte_s); rf_proba = rf.predict_proba(Xte_s)[:, 1]
    results['random_forest'] = {
        'accuracy': float(accuracy_score(yte, rf_pred)),
        'baseline_up_rate': float(yte.mean()),
        'next_day_prob_up': float(rf_proba[-1]),
        'feature_importances': {FEATURE_COLS[i]: float(rf.feature_importances_[i]) for i in range(len(FEATURE_COLS))},
    }
    return results

def train_lstm(df):
    prices = df['close'].dropna().values.reshape(-1, 1)
    sc = MinMaxScaler((0, 1)); scaled = sc.fit_transform(prices)
    X, y = [], []
    for i in range(LSTM_LOOKBACK, len(scaled)):
        X.append(scaled[i - LSTM_LOOKBACK:i, 0]); y.append(scaled[i, 0])
    X = np.array(X).reshape(-1, LSTM_LOOKBACK, 1); y = np.array(y)
    split = int(len(X) * 0.8)
    Xtr, Xte = X[:split], X[split:]; ytr, yte = y[:split], y[split:]
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=(LSTM_LOOKBACK, 1)), Dropout(0.2),
        LSTM(50, return_sequences=True), Dropout(0.2),
        LSTM(50, return_sequences=False), Dropout(0.2),
        Dense(25, activation='relu'), Dense(1),
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    early = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model.fit(Xtr, ytr, batch_size=32, epochs=50, validation_data=(Xte, yte), callbacks=[early], verbose=0)
    pred = sc.inverse_transform(model.predict(Xte, verbose=0)).flatten()
    actual = sc.inverse_transform(yte.reshape(-1, 1)).flatten()
    rmse = float(np.sqrt(mean_squared_error(actual, pred)))
    mae = float(mean_absolute_error(actual, pred))
    mape = float(mean_absolute_percentage_error(actual, pred) * 100)
    last = scaled[-LSTM_LOOKBACK:].reshape(1, LSTM_LOOKBACK, 1)
    forecast = []
    cur = last.copy()
    for _ in range(LSTM_FORECAST_DAYS):
        nxt = model.predict(cur, verbose=0)[0, 0]
        forecast.append(nxt)
        cur = np.concatenate([cur[:, 1:, :], [[[nxt]]]], axis=1)
    forecast = sc.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten().tolist()
    return {
        'rmse': rmse, 'mae': mae, 'mape': mape,
        'test_actual': actual.tolist(), 'test_predicted': pred.tolist(),
        'forecast_30d': forecast, 'lookback': LSTM_LOOKBACK,
    }

def train_arima(df):
    prices = df['close'].dropna().tail(500).reset_index(drop=True)
    split = int(len(prices) * 0.8)
    train, test = prices.iloc[:split], prices.iloc[split:]
    history = list(train); test_pred = []
    for actual in test:
        try:
            m = ARIMA(history, order=ARIMA_ORDER).fit()
            yhat = float(m.forecast()[0])
        except Exception:
            yhat = history[-1]
        test_pred.append(yhat); history.append(actual)
    test_actual = test.tolist()
    rmse = float(np.sqrt(mean_squared_error(test_actual, test_pred)))
    mae = float(mean_absolute_error(test_actual, test_pred))
    mape = float(mean_absolute_percentage_error(test_actual, test_pred) * 100)
    full = ARIMA(list(prices), order=ARIMA_ORDER).fit()
    forecast = [float(v) for v in full.forecast(ARIMA_FORECAST_DAYS)]
    return {
        'order': list(ARIMA_ORDER),
        'rmse': rmse, 'mae': mae, 'mape': mape,
        'test_actual': test_actual, 'test_predicted': test_pred,
        'forecast_30d': forecast, 'aic': float(full.aic),
    }

def multi_horizon_signals(df):
    out = {}
    for h in (1, 3, 5, 10, 20):
        d = df.copy()
        d['target'] = (d['close'].shift(-h) > d['close']).astype(int)
        d = d.dropna(subset=FEATURE_COLS + ['target'])
        if len(d) < 200:
            out[str(h)] = None; continue
        split = int(len(d) * 0.8)
        Xtr, Xte = d[FEATURE_COLS].iloc[:split].values, d[FEATURE_COLS].iloc[split:].values
        ytr, yte = d['target'].iloc[:split].values, d['target'].iloc[split:].values
        sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
        rf = RandomForestClassifier(n_estimators=80, random_state=42, max_depth=8).fit(Xtr, ytr)
        last_row = sc.transform(d[FEATURE_COLS].iloc[[-1]].values)
        out[str(h)] = {
            'horizon_days': h,
            'accuracy': float(accuracy_score(yte, rf.predict(Xte))),
            'p_up': float(rf.predict_proba(last_row)[0, 1]),
        }
    return out

def build_forecast_payload(ticker):
    df_raw = fetch_stock_data(ticker)
    df = engineer_features(df_raw)
    return {
        'ticker': ticker,
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'data_range': {
            'start': df_raw['date'].min().strftime('%Y-%m-%d'),
            'end': df_raw['date'].max().strftime('%Y-%m-%d'),
            'trading_days': int(len(df_raw)),
        },
        'last_close': float(df_raw['close'].iloc[-1]),
        'classifiers': train_classifiers(df),
        'lstm': train_lstm(df_raw),
        'arima': train_arima(df_raw),
        'horizons': multi_horizon_signals(df),
    }

In [ ]:
forecast_files = []
forecasts_manifest_tickers = []

if RUN_FORECASTS:
    print(f'Training models for {len(SYMBOLS)} tickers — this will take a while.')
    t0 = time.time()
    for i, sym in enumerate(SYMBOLS, 1):
        t1 = time.time()
        try:
            payload = build_forecast_payload(sym)
            content = json.dumps(payload)
            (FORECASTS_DIR / f'{sym}.json').write_text(content)
            forecast_files.append((f'data/forecasts/{sym}.json', content))
            forecasts_manifest_tickers.append(sym)
            print(f'  [{i:3}/{len(SYMBOLS)}] {sym:6} done in {time.time()-t1:.0f}s')
        except Exception as e:
            print(f'  [{i:3}/{len(SYMBOLS)}] {sym:6} FAILED: {e}')

    manifest = {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'tickers': forecasts_manifest_tickers,
    }
    manifest_content = json.dumps(manifest, indent=2)
    (FORECASTS_DIR / 'manifest.json').write_text(manifest_content)
    forecast_files.append(('data/forecasts/manifest.json', manifest_content))

    elapsed = time.time() - t0
    print(f'\nForecasts: trained {len(forecasts_manifest_tickers)}/{len(SYMBOLS)} tickers in {elapsed/60:.1f} min')
else:
    print('Skipping forecasts (RUN_FORECASTS=False).')

## 6. Push to GitHub

All generated JSONs are batched into a single Git commit via the GitHub API. This is much faster than committing files one by one, and GitHub Pages only redeploys once at the end.

If `PUSH_TO_GITHUB` is `False` (set in the config cell), this cell is a no-op — useful for inspecting the JSONs locally first.

In [ ]:
if not PUSH_TO_GITHUB:
    print('PUSH_TO_GITHUB=False — skipping push. Generated files live under /content/out/data/ in this Colab.')
else:
    all_files = bars_files + forecast_files
    if not all_files:
        print('Nothing to push.')
    else:
        print(f'Pushing {len(all_files)} files to {repo.full_name} ({GH_BRANCH})...')

        # Build a single tree containing all blobs, parented on the current HEAD,
        # then update the branch ref to point at the new commit.
        branch_ref = repo.get_git_ref(f'heads/{GH_BRANCH}')
        base_commit = repo.get_git_commit(branch_ref.object.sha)
        base_tree = base_commit.tree

        elements = []
        for path, content in all_files:
            blob = repo.create_git_blob(content, 'utf-8')
            elements.append(InputGitTreeElement(
                path=path, mode='100644', type='blob', sha=blob.sha,
            ))

        new_tree = repo.create_git_tree(elements, base_tree)

        what = []
        if bars_files: what.append(f'{len(bars_manifest_tickers)} bars')
        if forecast_files: what.append(f'{len(forecasts_manifest_tickers)} forecasts')
        msg = 'Refresh data: ' + ', '.join(what) + f' @ {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}'

        new_commit = repo.create_git_commit(msg, new_tree, [base_commit])
        branch_ref.edit(new_commit.sha)

        print(f'\nPushed: {new_commit.sha[:7]}')
        print(f'  message: {msg}')
        print(f'  https://github.com/{GH_USER}/{GH_REPO}/commit/{new_commit.sha}')
        print('\nGitHub Pages will redeploy in ~30 seconds.')

## Done

Once Pages redeploys, your live dashboard at `https://<user>.github.io/quantis/` will reflect the new data. The dashboard's methodology page reads each ticker's `generated_at` field to show when the data was refreshed.

**Next refresh:** just re-run this whole notebook. Bars-only refresh is fast (~2 min) and a good weekly cadence. Full forecasts retrain can wait for monthly or whenever the market regime shifts.